# Analyse TPS — Socle VOP Unitaire → Circuit RVM

**Objectif :** Mesurer les fluctuations d'émission TPS du socle unitaire vers RVM  
pour calibrer l'émission du socle masse sans dépasser le plafond imposé par RVM.

**Période :** Depuis le 9 octobre 2025 | **Granularité :** Tranches de 5 secondes

---

## 1. Configuration

In [ ]:
import dataiku
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

PLAFOND_TPS_RVM = 50   # ⚠️ À ajuster selon valeur contractuelle RVM

C_BLUE   = '#2196F3'
C_RED    = '#E63946'
C_ORANGE = '#FF9800'
C_GREEN  = '#2ECC71'
C_HEAD   = '#1A1A2E'
C_BG     = '#FAFBFC'
FONT     = 'Inter, Arial, sans-serif'

print(f'Plafond TPS RVM configuré : {PLAFOND_TPS_RVM} TPS  ← à confirmer avec RVM')

## 2. Chargement des données

In [ ]:
DATASET_NAME = 'NOM_DATASET_SOCLE_UNITAIRE'  # ← à remplacer

df_raw = dataiku.Dataset(DATASET_NAME).get_dataframe()
df_raw.columns = [c.strip().lower() for c in df_raw.columns]

df_raw['date']        = pd.to_datetime(df_raw['date'], errors='coerce').dt.date
df_raw['nb_requetes'] = pd.to_numeric(df_raw['nb_requetes'], errors='coerce')
for c in ['heure','minute','seconde']:
    if c in df_raw.columns:
        df_raw[c] = pd.to_numeric(df_raw[c], errors='coerce')

df_raw = df_raw.dropna(subset=['date','heure','minute','seconde','nb_requetes'])

df = df_raw[
    (df_raw['circuit'] == 'RVM') &
    (df_raw['date'] >= pd.to_datetime('2025-10-09').date())
].copy()

print(f'Lignes    : {len(df):,}')
print(f'Période   : {df.date.min()} → {df.date.max()}')
print(f'Nb jours  : {df.date.nunique()}')

## 3. Calcul TPS

In [ ]:
df['tps'] = df['nb_requetes'] / 5.0

df['ts'] = pd.to_datetime(
    df['date'].astype(str) + ' ' +
    df['heure'].astype(int).astype(str).str.zfill(2) + ':' +
    df['minute'].astype(int).astype(str).str.zfill(2) + ':' +
    df['seconde'].astype(int).astype(str).str.zfill(2)
)
df['heure_int']        = df['heure'].astype(int)
df['taux_utilisation'] = (df['tps'] / PLAFOND_TPS_RVM * 100).round(2)
df['depassement']      = df['tps'] > PLAFOND_TPS_RVM
df = df.sort_values('ts').reset_index(drop=True)

p95     = df['tps'].quantile(0.95)
p99     = df['tps'].quantile(0.99)
tps_max = df['tps'].max()
tps_moy = df['tps'].mean()
tps_med = df['tps'].median()
nb_dep  = int(df['depassement'].sum())

print(f"TPS max       : {tps_max:.3f}")
print(f"TPS P99       : {p99:.3f}")
print(f"TPS P95       : {p95:.3f}")
print(f"TPS médian    : {tps_med:.3f}")
print(f"TPS moyen     : {tps_moy:.3f}")
print(f"Dépassements  : {nb_dep:,}")

## 4. KPIs synthétiques

In [ ]:
kpi_data = [
    ('TPS Maximum',       tps_max,                    C_RED),
    ('TPS P99',           p99,                         '#FF5722'),
    ('TPS P95',           p95,                         C_ORANGE),
    ('TPS Médian',        tps_med,                     C_BLUE),
    ('TPS Moyen',         tps_moy,                     C_BLUE),
    ('Plafond RVM',       PLAFOND_TPS_RVM,             C_HEAD),
    ('Utilisation max %', df.taux_utilisation.max(),   C_RED),
    ('Utilisation moy %', df.taux_utilisation.mean(),  C_ORANGE),
    ('Nb dépassements',   nb_dep,                      C_RED),
    ('Nb jours analysés', df.date.nunique(),           C_HEAD),
]

fig = make_subplots(rows=2, cols=5, specs=[[{'type':'indicator'}]*5]*2)
for i, (lbl, val, clr) in enumerate(kpi_data):
    r, c = i//5+1, i%5+1
    fig.add_trace(go.Indicator(
        mode='number',
        value=float(val),
        number={'font':{'color':clr,'size':30,'family':FONT},
                'valueformat':'.2f' if i < 8 else '.0f'},
        title={'text':f'<b style="font-size:12px;color:{C_HEAD}">{lbl}</b>',
               'font':{'family':FONT}}
    ), row=r, col=c)

fig.update_layout(
    title=dict(text='<b>KPIs TPS — Socle Unitaire VOP → RVM</b>',
               font=dict(size=15,color=C_HEAD,family=FONT), x=0.01),
    paper_bgcolor=C_BG, height=260,
    margin=dict(l=10,r=10,t=60,b=10)
)
fig.show(renderer='notebook')

## 5. Jauges — Taux d'utilisation du plafond RVM

In [ ]:
util_max = round(tps_max / PLAFOND_TPS_RVM * 100, 1)
util_p99 = round(p99     / PLAFOND_TPS_RVM * 100, 1)
util_moy = round(tps_moy / PLAFOND_TPS_RVM * 100, 1)

fig = make_subplots(rows=1, cols=3,
    specs=[[{'type':'indicator'}]*3],
    subplot_titles=[
        f'Pic absolu ({tps_max:.2f} TPS)',
        f'P99 ({p99:.2f} TPS)',
        f'Moyenne ({tps_moy:.2f} TPS)'
    ])

for i, (val, clr, tps_val) in enumerate([
    (util_max, C_RED,    tps_max),
    (util_p99, C_ORANGE, p99),
    (util_moy, C_BLUE,   tps_moy),
]):
    restant = PLAFOND_TPS_RVM - tps_val
    fig.add_trace(go.Indicator(
        mode='gauge+number',
        value=val,
        number={'suffix':'%', 'font':{'size':28,'family':FONT,'color':clr}},
        gauge={
            'axis':{'range':[0,100],'ticksuffix':'%',
                    'tickvals':[0,20,40,60,80,100],
                    'tickfont':{'size':10,'family':FONT}},
            'bar':{'color':clr,'thickness':0.25},
            'bgcolor':'white',
            'steps':[
                {'range':[0,60],  'color':'#E8F5E9'},
                {'range':[60,80], 'color':'#FFF9C4'},
                {'range':[80,100],'color':'#FFEBEE'},
            ],
            'threshold':{'line':{'color':C_RED,'width':3},'thickness':0.8,'value':80}
        },
        title={
            'text': (f'<b style="color:{C_HEAD}">TPS = {tps_val:.2f}</b><br>'
                     f'<span style="font-size:11px;color:#64748B">'
                     f'Reste {restant:.2f} TPS pour la masse</span>'),
            'font':{'size':12,'family':FONT}
        }
    ), row=1, col=i+1)

fig.update_layout(
    title=dict(
        text=('<b>Taux d'utilisation du plafond RVM</b><br>'
              '<span style="font-size:11px;color:#64748B">'
              '🟢 &lt; 60% — OK · 🟡 60–80% — Attention · 🔴 &gt; 80% — Critique</span>'),
        font=dict(size=15,color=C_HEAD,family=FONT), x=0.01),
    paper_bgcolor=C_BG, height=340,
    margin=dict(l=20,r=20,t=100,b=20)
)
fig.show(renderer='notebook')

## 6. Fluctuations TPS — Toute la période

In [ ]:
df_min = df.set_index('ts')['tps'].resample('1min').max().reset_index()
df_min.columns = ['ts','tps']
df_min['util_pct'] = (df_min['tps'] / PLAFOND_TPS_RVM * 100).round(1)
df_min['statut']   = df_min['tps'].apply(
    lambda x: '🔴 Dépassement' if x > PLAFOND_TPS_RVM
    else ('🟡 Attention' if x > PLAFOND_TPS_RVM * 0.8 else '🟢 Normal'))

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_min['ts'], y=df_min['tps'],
    mode='lines', name='TPS max / minute',
    line=dict(color=C_BLUE, width=1.2),
    fill='tozeroy', fillcolor='rgba(33,150,243,0.07)',
    customdata=np.stack([df_min['util_pct'], df_min['statut']], axis=-1),
    hovertemplate=(
        '<b>%{x|%A %d/%m/%Y %H:%M}</b><br>'
        'TPS : <b>%{y:.2f}</b> req/s<br>'
        'Utilisation plafond : <b>%{customdata[0]:.1f}%</b><br>'
        'Statut : %{customdata[1]}<br>'
        f'Plafond RVM : {PLAFOND_TPS_RVM} TPS'
        '<extra></extra>'
    )
))

fig.add_hline(y=PLAFOND_TPS_RVM, line_dash='dash', line_color=C_RED, line_width=2,
              annotation_text=f'⛔ Plafond RVM : {PLAFOND_TPS_RVM} TPS',
              annotation_position='top left',
              annotation_font=dict(color=C_RED, size=11, family=FONT))

fig.add_hline(y=PLAFOND_TPS_RVM * 0.8, line_dash='dot', line_color=C_ORANGE, line_width=1.5,
              annotation_text=f'⚠️ Seuil alerte 80% : {PLAFOND_TPS_RVM*0.8:.0f} TPS',
              annotation_position='bottom right',
              annotation_font=dict(color=C_ORANGE, size=10, family=FONT))

fig.add_hline(y=p99, line_dash='dot', line_color='#9C27B0', line_width=1.5,
              annotation_text=f'P99 : {p99:.2f} TPS',
              annotation_position='top right',
              annotation_font=dict(color='#9C27B0', size=10, family=FONT))

fig.update_layout(
    title=dict(text='<b>Fluctuations TPS — Socle Unitaire → RVM</b><br>'
                    '<span style="font-size:11px;color:#64748B">TPS max par minute · Survolez pour le détail</span>',
               font=dict(size=15,color=C_HEAD,family=FONT), x=0.01),
    xaxis=dict(title='Date / Heure', showgrid=True, gridcolor='#F0F4F8',
               tickfont=dict(size=10,family=FONT), rangeslider=dict(visible=True)),
    yaxis=dict(title='TPS (req/s)', showgrid=True, gridcolor='#F0F4F8',
               tickfont=dict(size=10,family=FONT)),
    paper_bgcolor=C_BG, plot_bgcolor='white',
    hovermode='x unified', height=480,
    margin=dict(l=60,r=30,t=90,b=80)
)
fig.show(renderer='notebook')

## 7. Marge disponible pour le socle Masse

In [ ]:
marge_secu       = round(PLAFOND_TPS_RVM * 0.05, 2)
tps_unitaire_p99 = round(p99, 2)
tps_masse_dispo  = round(max(0, PLAFOND_TPS_RVM - tps_unitaire_p99 - marge_secu), 2)

fig = go.Figure()

fig.add_trace(go.Bar(
    name=f'Unitaire P99',
    x=['Plafond RVM'], y=[tps_unitaire_p99],
    marker_color=C_BLUE,
    text=[f'{tps_unitaire_p99} TPS'],
    textposition='inside', textfont=dict(color='white',size=13,family=FONT),
    customdata=[[tps_unitaire_p99, round(tps_unitaire_p99/PLAFOND_TPS_RVM*100,1)]],
    hovertemplate=(
        '<b>Socle Unitaire (P99)</b><br>'
        'TPS consommé : <b>%{customdata[0]:.2f} TPS</b><br>'
        'Part du plafond : <b>%{customdata[1]:.1f}%</b><br>'
        'Interprétation : occupation actuelle du socle unitaire'
        '<extra></extra>'
    )
))

fig.add_trace(go.Bar(
    name=f'Disponible masse',
    x=['Plafond RVM'], y=[tps_masse_dispo],
    marker_color=C_GREEN,
    text=[f'{tps_masse_dispo} TPS'],
    textposition='inside', textfont=dict(color='white',size=13,family=FONT),
    customdata=[[tps_masse_dispo, round(tps_masse_dispo/PLAFOND_TPS_RVM*100,1),
                 round(1/tps_masse_dispo,3) if tps_masse_dispo>0 else 0]],
    hovertemplate=(
        '<b>Bande passante disponible — Masse</b><br>'
        'TPS disponible : <b>%{customdata[0]:.2f} TPS</b><br>'
        'Part du plafond : <b>%{customdata[1]:.1f}%</b><br>'
        'Fréquence max : <b>1 req / %{customdata[2]:.3f}s</b>'
        '<extra></extra>'
    )
))

fig.add_trace(go.Bar(
    name=f'Marge sécurité 5%',
    x=['Plafond RVM'], y=[marge_secu],
    marker_color=C_ORANGE,
    text=[f'{marge_secu} TPS'],
    textposition='inside', textfont=dict(color='white',size=13,family=FONT),
    customdata=[[marge_secu]],
    hovertemplate=(
        '<b>Marge de sécurité</b><br>'
        'Réservée : <b>%{customdata[0]:.2f} TPS</b><br>'
        'Objectif : absorber les pics imprévus'
        '<extra></extra>'
    )
))

fig.update_layout(
    barmode='stack',
    title=dict(
        text=(f'<b>Répartition du plafond RVM ({PLAFOND_TPS_RVM} TPS)</b><br>'
              '<span style="font-size:11px;color:#64748B">'
              '🔵 Unitaire · 🟢 Disponible masse · 🟠 Marge sécurité — Survolez pour le détail</span>'),
        font=dict(size=15,color=C_HEAD,family=FONT), x=0.01),
    xaxis=dict(tickfont=dict(size=12,family=FONT)),
    yaxis=dict(title='TPS (req/s)', range=[0, PLAFOND_TPS_RVM*1.1],
               showgrid=True, gridcolor='#F0F4F8',
               tickfont=dict(size=11,family=FONT)),
    paper_bgcolor=C_BG, plot_bgcolor='white',
    legend=dict(orientation='h', y=-0.2, font=dict(size=11,family=FONT)),
    height=430, margin=dict(l=60,r=30,t=100,b=80)
)
fig.show(renderer='notebook')

## 8. Distribution des TPS

In [ ]:
fig = go.Figure()
fig.add_trace(go.Histogram(
    x=df['tps'], nbinsx=60, name='Nb tranches',
    marker_color=C_BLUE, marker_line_color='white', marker_line_width=0.5,
    opacity=0.85,
    hovertemplate=(
        'TPS : <b>%{x:.1f} req/s</b><br>'
        'Nb tranches de 5s : <b>%{y:,}</b><br>'
        '<extra></extra>'
    )
))

for val, lbl, clr, desc in [
    (p95,     f'P95 : {p95:.2f}',    C_ORANGE, '95% des tranches sont en-dessous'),
    (p99,     f'P99 : {p99:.2f}',    C_RED,    '99% des tranches sont en-dessous'),
    (tps_max, f'Max : {tps_max:.2f}','#9C27B0','Pic absolu observé'),
]:
    fig.add_vline(x=val, line_dash='dash', line_color=clr, line_width=2,
                  annotation_text=lbl, annotation_position='top',
                  annotation_font=dict(color=clr, size=11, family=FONT))

fig.update_layout(
    title=dict(
        text=('<b>Distribution des TPS observés — Socle Unitaire → RVM</b><br>'
              '<span style="font-size:11px;color:#64748B">Survolez les barres pour le détail · Lignes = percentiles clés</span>'),
        font=dict(size=15,color=C_HEAD,family=FONT), x=0.01),
    xaxis=dict(title='TPS (req/s)', showgrid=True, gridcolor='#F0F4F8',
               tickfont=dict(size=10,family=FONT)),
    yaxis=dict(title='Nb tranches de 5s', showgrid=True, gridcolor='#F0F4F8',
               tickfont=dict(size=10,family=FONT)),
    paper_bgcolor=C_BG, plot_bgcolor='white',
    height=400, margin=dict(l=60,r=30,t=90,b=60)
)
fig.show(renderer='notebook')

## 9. Heatmap TPS P95 — heure × jour de la semaine

In [ ]:
df['jour_semaine'] = pd.to_datetime(df['date'].astype(str)).dt.day_name()
hm = (df.groupby(['jour_semaine','heure_int'])['tps']
        .quantile(0.95).reset_index())
hm.columns = ['Jour','Heure','TPS_P95']

ordre  = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
fr_map = {'Monday':'Lundi','Tuesday':'Mardi','Wednesday':'Mercredi',
          'Thursday':'Jeudi','Friday':'Vendredi','Saturday':'Samedi','Sunday':'Dimanche'}
hm['Jour'] = pd.Categorical(hm['Jour'], categories=ordre, ordered=True)
hm = hm.sort_values(['Jour','Heure'])
hm['Jour_fr'] = hm['Jour'].map(fr_map)
hm['util_pct'] = (hm['TPS_P95'] / PLAFOND_TPS_RVM * 100).round(1)

pivot      = hm.pivot(index='Jour_fr', columns='Heure', values='TPS_P95')
pivot_util = hm.pivot(index='Jour_fr', columns='Heure', values='util_pct')
pivot      = pivot.reindex([v for v in fr_map.values() if v in pivot.index])
pivot_util = pivot_util.reindex(pivot.index)

fig = px.imshow(
    pivot, color_continuous_scale='Blues',
    labels=dict(x='Heure', y='', color='TPS P95'),
    aspect='auto',
    zmin=0, zmax=PLAFOND_TPS_RVM
)

# Hover enrichi via customdata
customdata_arr = np.dstack([pivot.values, pivot_util.values])
fig.update_traces(
    customdata=customdata_arr,
    hovertemplate=(
        '<b>%{y} — %{x}h00</b><br>'
        'TPS P95 : <b>%{customdata[0]:.2f} req/s</b><br>'
        'Utilisation plafond : <b>%{customdata[1]:.1f}%</b><br>'
        f'Plafond RVM : {PLAFOND_TPS_RVM} TPS'
        '<extra></extra>'
    )
)

fig.update_layout(
    title=dict(
        text=('<b>Heatmap TPS P95 — par heure et jour de la semaine</b><br>'
              '<span style="font-size:11px;color:#64748B">'
              'Plus c'est foncé = plus c'est chargé · Survolez pour le détail</span>'),
        font=dict(size=15,color=C_HEAD,family=FONT), x=0.01),
    xaxis=dict(title='Heure', tickmode='linear', dtick=1,
               tickfont=dict(size=10,family=FONT)),
    yaxis=dict(tickfont=dict(size=11,family=FONT)),
    coloraxis_colorbar=dict(title='TPS P95', tickfont=dict(size=10,family=FONT)),
    paper_bgcolor=C_BG, height=380,
    margin=dict(l=80,r=80,t=100,b=60)
)
fig.show(renderer='notebook')

## 10. Top 10 — Jours les plus chargés

In [ ]:
top10 = (
    df.groupby('date')['tps']
      .agg(tps_max='max',
           tps_p99=lambda x: x.quantile(0.99),
           tps_p95=lambda x: x.quantile(0.95),
           tps_moy='mean',
           nb_dep=lambda x: (x > PLAFOND_TPS_RVM).sum())
      .reset_index()
      .sort_values('tps_max', ascending=False)
      .head(10)
      .reset_index(drop=True)
)
top10['util_pct']  = (top10['tps_max'] / PLAFOND_TPS_RVM * 100).round(1)
top10['date_str']  = pd.to_datetime(top10['date'].astype(str)).dt.strftime('%d/%m/%Y')
top10['jour_sem']  = pd.to_datetime(top10['date'].astype(str)).dt.day_name().map(fr_map)

def clr_bar(u):
    if u >= 80: return C_RED
    if u >= 60: return C_ORANGE
    return C_BLUE

colors = [clr_bar(u) for u in top10['util_pct']]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=top10['date_str'],
    y=top10['tps_max'],
    marker_color=colors,
    text=[f"{v:.1f}<br>({u:.0f}%)" for v, u in zip(top10['tps_max'], top10['util_pct'])],
    textposition='outside',
    textfont=dict(size=10, family=FONT),
    customdata=np.stack([
        top10['tps_max'], top10['tps_p99'], top10['tps_p95'],
        top10['tps_moy'], top10['util_pct'], top10['nb_dep'],
        top10['jour_sem']
    ], axis=-1),
    hovertemplate=(
        '<b>%{x} — %{customdata[6]}</b><br>'
        '─────────────────────<br>'
        'TPS max  : <b>%{customdata[0]:.2f} req/s</b><br>'
        'TPS P99  : <b>%{customdata[1]:.2f} req/s</b><br>'
        'TPS P95  : <b>%{customdata[2]:.2f} req/s</b><br>'
        'TPS moy  : <b>%{customdata[3]:.2f} req/s</b><br>'
        'Utilisation : <b>%{customdata[4]:.1f}%</b> du plafond<br>'
        'Dépassements : <b>%{customdata[5]:.0f}</b> tranches'
        '<extra></extra>'
    )
))

fig.add_hline(y=PLAFOND_TPS_RVM, line_dash='dash', line_color=C_RED, line_width=2,
              annotation_text=f'Plafond {PLAFOND_TPS_RVM} TPS',
              annotation_font=dict(color=C_RED, size=11, family=FONT))

fig.update_layout(
    title=dict(
        text=('<b>Top 10 — Journées avec TPS maximum le plus élevé</b><br>'
              '<span style="font-size:11px;color:#64748B">'
              '🔴 ≥ 80% · 🟠 60–80% · 🔵 &lt; 60% du plafond · Survolez pour le détail complet</span>'),
        font=dict(size=15,color=C_HEAD,family=FONT), x=0.01),
    xaxis=dict(tickfont=dict(size=10,family=FONT)),
    yaxis=dict(title='TPS max (req/s)', showgrid=True, gridcolor='#F0F4F8',
               tickfont=dict(size=10,family=FONT)),
    paper_bgcolor=C_BG, plot_bgcolor='white',
    showlegend=False, height=450,
    margin=dict(l=60,r=30,t=100,b=60)
)
fig.show(renderer='notebook')

## 11. Distribution TPS par heure

In [ ]:
stats_h = (df.groupby('heure_int')['tps']
             .agg(q25=lambda x: x.quantile(0.25),
                  median='median',
                  q75=lambda x: x.quantile(0.75),
                  p95=lambda x: x.quantile(0.95),
                  p99=lambda x: x.quantile(0.99),
                  tmax='max',
                  tmoy='mean')
             .reset_index())

fig = go.Figure()
for _, row in stats_h.iterrows():
    h = int(row['heure_int'])
    sub = df[df['heure_int']==h]['tps']
    fig.add_trace(go.Box(
        y=sub, name=f'{h:02d}h',
        marker_color=C_BLUE, line_color=C_HEAD,
        boxmean='sd',
        customdata=[[row['q25'], row['median'], row['q75'],
                     row['p95'], row['p99'], row['tmax'], row['tmoy'],
                     round(row['tmax']/PLAFOND_TPS_RVM*100,1)]],
        hovertemplate=(
            f'<b>{h:02d}h00</b><br>'
            '─────────────────<br>'
            'Q25   : %{customdata[0]:.2f} TPS<br>'
            'Médiane : %{customdata[1]:.2f} TPS<br>'
            'Q75   : %{customdata[2]:.2f} TPS<br>'
            'P95   : %{customdata[3]:.2f} TPS<br>'
            'P99   : %{customdata[4]:.2f} TPS<br>'
            'Max   : <b>%{customdata[5]:.2f} TPS</b><br>'
            'Moyen : %{customdata[6]:.2f} TPS<br>'
            'Utilisation max : <b>%{customdata[7]:.1f}%</b>'
            '<extra></extra>'
        )
    ))

fig.add_hline(y=PLAFOND_TPS_RVM, line_dash='dash', line_color=C_RED, line_width=2,
              annotation_text=f'⛔ Plafond RVM {PLAFOND_TPS_RVM} TPS',
              annotation_font=dict(color=C_RED, size=11, family=FONT))

fig.update_layout(
    title=dict(
        text=('<b>Distribution TPS par heure — Socle Unitaire → RVM</b><br>'
              '<span style="font-size:11px;color:#64748B">'
              'Survolez chaque boîte pour Q25/médiane/Q75/P95/P99/max</span>'),
        font=dict(size=15,color=C_HEAD,family=FONT), x=0.01),
    xaxis=dict(title='Heure', showgrid=False, tickfont=dict(size=10,family=FONT)),
    yaxis=dict(title='TPS (req/s)', showgrid=True, gridcolor='#F0F4F8',
               tickfont=dict(size=10,family=FONT)),
    paper_bgcolor=C_BG, plot_bgcolor='white',
    showlegend=False, height=480,
    margin=dict(l=60,r=30,t=90,b=60)
)
fig.show(renderer='notebook')

## 12. Tableau récapitulatif journalier

In [ ]:
recap = (
    df.groupby('date')['tps']
      .agg(tps_max='max',
           tps_p99=lambda x: x.quantile(0.99),
           tps_p95=lambda x: x.quantile(0.95),
           tps_moyen='mean',
           tps_median='median')
      .reset_index()
)
idx_max = df.groupby('date')['tps'].idxmax()
recap['heure_pic']       = df.loc[idx_max,'ts'].dt.strftime('%H:%M:%S').values
recap['depassements']    = df.groupby('date')['depassement'].sum().values
recap['utilisation_pct'] = (recap['tps_max'] / PLAFOND_TPS_RVM * 100).round(1)
for c in ['tps_max','tps_p99','tps_p95','tps_moyen','tps_median']:
    recap[c] = recap[c].round(3)
display(recap.sort_values('tps_max', ascending=False).reset_index(drop=True))

## 13. Recommandation — Calibrage émission Socle Masse

In [ ]:
tps_dispo_masse    = max(0, PLAFOND_TPS_RVM - p99)
tps_recommande     = min(tps_dispo_masse, PLAFOND_TPS_RVM * 0.80)
freq_interrogation = 1 / tps_recommande if tps_recommande > 0 else float('inf')

print('=' * 60)
print('   CALIBRAGE RECOMMANDÉ — ÉMISSION SOCLE MASSE → RVM')
print('=' * 60)
print(f'  Plafond TPS RVM contractuel     : {PLAFOND_TPS_RVM:.0f} TPS')
print(f'  TPS unitaire P99 observé        : {p99:.3f} TPS')
print(f'  Bande passante disponible       : {tps_dispo_masse:.3f} TPS')
print(f'  TPS recommandé pour la masse    : {tps_recommande:.3f} TPS')
print(f'  Fréquence d\'interrogation max   : 1 req / {freq_interrogation:.3f}s')
print('=' * 60)
print()
print('  Formule : TPS_masse = min(Plafond − P99_unitaire, Plafond × 80%)')
print()
print('  ⚠️  À valider avec Hervé et la valeur contractuelle RVM.')